# Delegate to another agent

This notebook deploys `research-coordinator`, an agent that delegates web research to `web-researcher`. See the [README](./README.md) for the same steps in a terminal.

kagent and MCP servers must be enabled on your prokube cluster.

## Setup

Set up this repository's kagent helpers from [kagent_helpers.py](../shared/kagent_helpers.py). Adding their directory to `sys.path` lets the notebook import them without installing anything.

In [ ]:
import sys

sys.path.insert(0, "../shared")  # kagent helpers (shared by all kagent examples)

import kagent_helpers

In [ ]:
import os
from getpass import getpass

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()

## Prerequisites

The coordinator needs `web-researcher` from [`agent-with-mcp`](../agent-with-mcp/) and everything it depends on. The next cell creates whatever is missing with helpers from [kagent_helpers.py](../shared/kagent_helpers.py) that run roughly the same commands as the earlier notebooks:

- `ensure_anthropic_secret()` creates the `anthropic-api-key` Secret, like the *Create the credentials* step of [basic-agent.ipynb](../basic-agent/basic-agent.ipynb). It skips a Secret that already exists, for example one created under **K8s Secrets** in the prokube user menu (see [Kubernetes Secrets](https://prokube.ai/docs/1.9/platform/kubernetes.html#kubernetes-secrets)), and prompts for the key unless `ANTHROPIC_API_KEY` is set.
- `ensure_model_config()` deploys the `anthropic-haiku` ModelConfig (Claude Haiku), like the *Deploy* step of [basic-agent.ipynb](../basic-agent/basic-agent.ipynb).
- `ensure_fetch_server()` deploys the Fetch MCP server, which fetches web pages, like the *Deploy* step of [fetch-server.ipynb](../../mcp-servers/deploy-upstream-mcp-server/fetch-server.ipynb).
- `ensure_web_researcher()` deploys `web-researcher`, like the *Deploy* step of [web-researcher.ipynb](../agent-with-mcp/web-researcher.ipynb). It first looks up the Fetch tool's name, `fetch` or `fetch-example_fetch`, because Agent Gateway prefixes tool names with the server name when the workspace has more than one MCP server.

In [ ]:
kagent_helpers.ensure_anthropic_secret()
kagent_helpers.ensure_model_config()
kagent_helpers.ensure_fetch_server()
kagent_helpers.ensure_web_researcher()

## How it works

The coordinator in [`research-coordinator.yaml`](./research-coordinator.yaml) lists `web-researcher` as a tool of type `Agent`:

```yaml
spec:
  declarative:
    tools:
    - type: Agent
      agent:
        name: web-researcher
```

kagent offers `web-researcher` to the coordinator's model as a tool. When the model calls it, kagent sends the request to `web-researcher` over A2A and returns its answer to the coordinator. The coordinator has no MCP tools of its own; its system message tells it to delegate web research.

## Deploy

In [ ]:
!kubectl apply -f research-coordinator.yaml
!kubectl wait --for=condition=Ready agents.kagent.dev/research-coordinator --timeout=3m

## Talk to the agent from this Lab

Labs reach agents through the internal Agent Gateway route without an API key. You can also open the [Agents page](/pkui/kagent) in the prokube UI (`https://<your-prokube-domain>/pkui/kagent`), select `research-coordinator`, and chat there.

Expect one sentence about what prokube offers that includes the source URL `https://prokube.ai`, as returned by `web-researcher`.

In [ ]:
internal_url = kagent_helpers.workspace_a2a_url(namespace, "research-coordinator")
print(internal_url)

In [ ]:
prompt = "Research https://prokube.ai and summarize what prokube offers in one sentence. Include the source URL."
print(kagent_helpers.send_message(internal_url, prompt))

## Connect with an API key

Clients outside the cluster use the agent's external route, `https://<your-prokube-domain>/svc/a2a/<workspace>/research-coordinator`, with an API key. See [Public vs. Internal Traffic](https://prokube.ai/docs/1.9/platform/agent_gateway.html#public-vs-internal-traffic) for how this route differs from the internal one.

Enter your prokube domain, the host name you open prokube at, for example `my-cluster.prokube.cloud`. The cell uses `PROKUBE_DOMAIN` instead if it is set:

In [ ]:
domain = os.environ.get("PROKUBE_DOMAIN") or input("prokube domain (e.g. my-cluster.prokube.cloud): ")

Create a key with access to `research-coordinator` on the [API Keys page](/pkui/ai-gateway/keys) (`https://<your-prokube-domain>/pkui/ai-gateway/keys`), then paste it when prompted below. The cell uses `A2A_API_KEY` instead if it is set:

In [ ]:
api_key = os.environ.get("A2A_API_KEY") or getpass("Agent API key: ")

Send the same prompt through the agent's external URL:

In [ ]:
api_key_url = f"https://{domain}/svc/a2a/{namespace}/research-coordinator"
print(api_key_url)

In [ ]:
print(kagent_helpers.send_message(api_key_url, prompt, api_key=api_key))

## Clean up

Uncomment the command and run the cell to delete `research-coordinator`. This leaves `web-researcher` and its prerequisites. To remove everything, also run the clean-up cells of the earlier notebooks and delete the Fetch server.

In [ ]:
# !kubectl delete -f research-coordinator.yaml